# 01 — Metric classes

Reads `reference/metric_classes.csv` and its provenance record, both produced by `scripts/01_build_metric_classes.py`, and asks what they imply for the pilot, the 250-claim annotation round that decides whether the full study is worth running.

Terms used below without explanation are defined in the [glossary](../docs/glossary.md).

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

pd.set_option("display.max_colwidth", 78)
pd.set_option("display.width", 200)

metrics = pd.read_csv(PROJECT_ROOT / "reference" / "metric_classes.csv")
provenance = json.loads((PROJECT_ROOT / "reference" / "metric_classes.provenance.json").read_text())
years = provenance["probe_years"]

print(f"{len(metrics)} metrics, built at commit {provenance['commit']} on {provenance['generated_at']}")
print(f"{provenance['counts']['elements_verified']} elements verified at {years}")

49 metrics, built at commit 8b474ee on 2026-08-30T10:11:33Z
36 elements verified at [2012, 2018, 2024]


## Class and evidence store

`class` decides the default baseline under section 5.4 of the [annotation guidelines](../docs/annotation-guidelines.md), which is what a claim's number is compared against when management does not say. A flow is measured over a period and compares to the same quarter a year earlier; a level is measured at a point in time and compares to the immediately prior quarter.

In [2]:
summary = pd.crosstab(metrics["class"], metrics["in_evidence_store"], margins=True, margins_name="total")
summary.columns.name = "in evidence store"
summary

in evidence store,no,yes,total
class,,,
FLOW,5,22,27
LEVEL,3,19,22
total,8,41,49


## Availability changes over the study window

An element that exists today need not have existed in 2012. ASC 606 introduced new revenue concepts around 2018 and began retiring their predecessors, so a single-period check would report an element as universally available when it covers only half the window.

The probe runs at three points across 2012–2024 for exactly this reason.

In [3]:
rows = []
for name, info in provenance["elements"].items():
    row = {"element": name.split(":")[-1], "coverage": info["window_coverage"]}
    for year, period in zip(years, sorted(info["periods"])):
        entry = info["periods"][period]
        row[year] = entry["filer_count"] if entry["exists"] else None
    rows.append(row)

availability = pd.DataFrame(rows).set_index("element")
availability[availability["coverage"] != "full"].sort_index()

,coverage,2012,2018,2024
element,,,,
ContractWithCustomerLiabilityCurrent,partial,NaN,543,1635
RevenueFromContractWithCustomerExcludingAssessedTax,partial,NaN,2304,2654
RevenueRemainingPerformanceObligation,partial,NaN,553,709


Three elements do not span the window, all of them ASC 606 concepts absent in 2012.

Two are recoverable, because the metric names a predecessor as an alternative and the pair covers the window between them. The third is not: remaining performance obligation was created by ASC 606 and has no pre-606 equivalent, so a 2013 claim about it cannot be settled at all.

In [4]:
alternatives = metrics[metrics["taxonomy_element"].str.contains(r"\|", na=False)]
alternatives[["metric", "window_coverage", "taxonomy_element"]].reset_index(drop=True)

,metric,window_coverage,taxonomy_element
0,revenue,full,us-gaap:Revenues | us-gaap:RevenueFromContractWithCustomerExcludingAssesse...
1,deferred_revenue,full,us-gaap:ContractWithCustomerLiabilityCurrent | us-gaap:DeferredRevenueCurrent


In [5]:
partial = metrics[metrics["window_coverage"] == "partial"]
partial[["metric", "class", "taxonomy_element", "note"]].reset_index(drop=True)

,metric,class,taxonomy_element,note
0,remaining_performance_obligation,LEVEL,us-gaap:RevenueRemainingPerformanceObligation,Mostly software and subscription filers.


## The metrics no evidence store can settle

These carry a class, because the baseline default applies regardless, but XBRL does not hold them at all. A claim naming one is checkable in principle and still gets the verdict NOT_ENOUGH_EVIDENCE, because the evidence store has nothing to check it against.

How often that happens is what `StructuredCoverage` measures: the share of claims the structured evidence store can actually settle. The pilot exists to measure it, and the answer decides how much of the study survives.

In [6]:
outside = metrics[metrics["in_evidence_store"] == "no"][["metric", "class", "note"]]
outside.reset_index(drop=True)

,metric,class,note
0,ebitda,FLOW,"Not a GAAP concept and never tagged. Derivable from tagged components, but..."
1,adjusted_ebitda,FLOW,Company-defined. Evidence availability is NON-GAAP-ONLY.
2,adjusted_eps,FLOW,Company-defined. Consensus estimates are usually struck on this rather tha...
3,constant_currency_revenue,FLOW,A transform on revenue rather than a separate metric. Record as revenue wi...
4,headcount,LEVEL,A count at a point in time. Disclosed in the 10-K cover or business sectio...
5,backlog,LEVEL,A level at a point in time. Rarely tagged. Where a filer reports remaining...
6,annual_recurring_revenue,LEVEL,Named as a revenue figure but reported as an annualised run rate at a poin...
7,bookings,FLOW,"Orders taken over a period, so FLOW. Not the same as backlog, which is the..."


## Thinly tagged elements

Existence is not availability. An element tagged by a few hundred filers is real, but a claim resolving to it will usually find nothing. These counts come from the SEC, at the most recent probed year.

In [7]:
latest = years[-1]
sparse = availability[availability[latest].notna() & (availability[latest] < 1000)]
print(f"median coverage in {latest}: {availability[latest].median():,.0f} filers")
sparse[[latest, "coverage"]].sort_values(latest)

median coverage in 2024: 2,892 filers


,2024,coverage
element,,
DebtCurrent,313,full
DeferredRevenueCurrent,433,full
RevenueRemainingPerformanceObligation,709,partial
PaymentsOfDividendsCommonStock,924,full


## Where the class assignment is arguable

Each was decided rather than left open, and each carries its reasoning. An annotator who disagrees logs it in `annotations/policy_gaps.md` rather than labelling against the [annotation guidelines](../docs/annotation-guidelines.md).

In [8]:
for _, row in metrics[metrics["ambiguous"] == "yes"].iterrows():
    print(f"{row['metric']}  [{row['class']}]")
    print(f"    {row['note']}\n")

gross_margin  [FLOW]
    A ratio of two flows, so FLOW, compared year over year. Arguable: for a non-seasonal filer management often compares sequentially. Year over year is correct for seasonal filers and merely conservative for the rest, which is why it wins.

operating_margin  [FLOW]
    A ratio of two flows, so FLOW, compared year over year. Arguable: for a non-seasonal filer management often compares sequentially. Year over year is correct for seasonal filers and merely conservative for the rest, which is why it wins.

net_margin  [FLOW]
    A ratio of two flows, so FLOW, compared year over year. Arguable: for a non-seasonal filer management often compares sequentially. Year over year is correct for seasonal filers and merely conservative for the rest, which is why it wins.

free_cash_flow  [FLOW]
    The class is unambiguous; the definition is not. Operating cash flow minus capex is the common form but filers vary, and some report it as non-GAAP. Record basis ADJUSTED_NON_GAAP wh

## What this implies for the pilot

**Eight of forty-nine metrics are outside the evidence store**, and five of those eight are flows: adjusted EBITDA, adjusted EPS, constant-currency revenue, EBITDA, bookings. That is the vocabulary of guidance. The metrics management most often makes forward-looking claims about are the ones XBRL cannot settle.

**Element availability is not constant across the window.** Three ASC 606 concepts are absent in 2012. Two are covered by naming their pre-606 predecessor as an alternative; remaining performance obligation has no predecessor, so claims about it before roughly 2018 are unadjudicable regardless of retrieval quality. Any coverage figure that ignores this will overstate the early years.

**Ten class assignments are arguable.** If per-field agreement on the baseline comes back below the demotion threshold, these rows are where to look, because they are where two careful annotators can follow the guidelines and still disagree.